In [1]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import os
import pandas as pd
from sklearn.mixture import GaussianMixture
from tqdm import tqdm
import matplotlib.pyplot as plt

In [10]:

class ImportanceSampling:
    def __init__(self, X, y, n_clusters=5, n_samples=1000, exploratory_fraction=0.5):
        """
        Initialize the ImportanceSampling object.

        Parameters:
        - X: ndarray, feature data.
        - y: ndarray, labels (0 or 1).
        - n_clusters: int, number of clusters for k-means.
        - n_samples: int, number of biased samples to generate.
        - exploratory_fraction: float, fraction of exploratory samples to add.
        """
        self.X = X
        self.y = y
        self.n_clusters = n_clusters
        self.n_samples = n_samples
        self.exploratory_fraction = exploratory_fraction

        # Internal attributes
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.X)
        self.kmeans = None
        self.X_signal = None
        self.biased_samples = None
        self.weights = None

    def apply_kmeans(self):
        """Apply k-means clustering to signal data (y=1)."""
        signal_indices = np.where(self.y == 1)[0]
        self.X_signal = self.X_scaled[signal_indices]
        self.kmeans = KMeans(n_clusters=self.n_clusters, n_init=10, random_state=42).fit(self.X_signal)

    def create_bias_distribution(self, scale=0.2):
        """Generate biased samples using Gaussian noise around k-means cluster centers."""
        cluster_centers = self.kmeans.cluster_centers_
        samples_per_cluster = np.random.multinomial(self.n_samples, [1 / self.n_clusters] * self.n_clusters)
        biased_samples = []
        for i, n in enumerate(samples_per_cluster):
            samples = np.random.normal(loc=cluster_centers[i], scale=scale, size=(n, self.X_signal.shape[1]))
            biased_samples.append(samples)
        self.biased_samples = np.vstack(biased_samples)

    def add_exploratory_samples(self):
        """Add exploratory samples to improve biasing distribution."""
        n_exploratory = int(self.n_samples * self.exploratory_fraction)
        exploratory_samples = np.random.uniform(low=self.X.min(axis=0), high=self.X.max(axis=0),
                                                size=(n_exploratory, self.X.shape[1]))
        exploratory_samples_scaled = self.scaler.transform(exploratory_samples)
        self.biased_samples = np.vstack([self.scaler.transform(self.biased_samples), exploratory_samples_scaled])

    def optimize_kde_bandwidth(self, data):
        """Optimize the bandwidth for KDE using cross-validation."""
        bandwidths = np.logspace(-1, 1, 20)
        grid = GridSearchCV(KernelDensity(kernel='gaussian'), {'bandwidth': bandwidths}, cv=3)
        grid.fit(data)
        return grid.best_params_['bandwidth']

    def fit_kde(self, bandwidth_p=None, bandwidth_q=None):
        """Fit KDE models for the target and biasing distributions."""
        # Optimize bandwidths
        if bandwidth_p==None:
            bandwidth_p = self.optimize_kde_bandwidth(self.X_scaled)
        if bandwidth_q==None:
            bandwidth_q = self.optimize_kde_bandwidth(self.biased_samples)

        # Fit KDE models
        self.kde_p = KernelDensity(bandwidth=bandwidth_p, kernel='gaussian').fit(self.X_scaled)
        self.kde_q = KernelDensity(bandwidth=bandwidth_q, kernel='gaussian').fit(self.biased_samples)

    def compute_weights(self):
        """Compute importance weights."""
        biased_samples_scaled = self.scaler.transform(self.biased_samples)
        log_p = self.kde_p.score_samples(biased_samples_scaled)
        log_q = self.kde_q.score_samples(biased_samples_scaled)

        plt.hist(log_p, bins=50, alpha=0.5, label='log p(x)')
        plt.hist(log_q, bins=50, alpha=0.5, label='log q(x)')
        plt.legend()
        plt.title("Log Densities of p(x) and q(x)")
        plt.show()

        # Regularize and compute weights
        log_diff = np.clip(log_p - log_q, -100, 100)
        self.weights = np.exp(log_diff)
        self.weights *= np.mean(self.y == 1)  # Scale weights by true signal ratio
        self.weights = np.nan_to_num(self.weights, nan=0.0, posinf=1.0, neginf=0.0)

        # Normalize weights
        if np.sum(self.weights) == 0:
            self.weights += 1e-10
        self.weights /= np.sum(self.weights)

    def estimate_probability(self):
        """Estimate the probability of y=1."""
        return np.sum(self.weights)

    def run(self,bandwidth_p=None, bandwidth_q=None):
        """Execute the entire importance sampling workflow."""
        self.apply_kmeans()
        self.create_bias_distribution()
        self.fit_kde(bandwidth_p,bandwidth_q)
        self.compute_weights()
        return self.estimate_probability()




In [3]:
version = 'vpce1.1'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
   
# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=["radius","thickness","npanels","theta","length","x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
y_label = 'nC_Ge77'

LF_noise = 0.028

file_in=f'Ge77_rates_CNP_{version}.csv'
data_train=pd.read_csv(f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/v1.4/tier2/neutron-sim-LF-v1.4-0001-tier2.csv')
row_lf=data_train.index[(data_train[y_label] == 1)]

x_lf = data_train.loc[row_lf][x_labels].to_numpy()
y_lf = data_train.loc[row_lf][ y_label].to_numpy()

In [4]:
# Loop through files
x_lf_list=[]
y_lf_list=[]
x_lf_list_sig=[]
for i in tqdm(range(300)):
    # Construct the file name
    file_in = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/v1.4/tier2/neutron-sim-LF-v1.4-{i:04}-tier2.csv'
    
    try:
        # Read the file
        data_train = pd.read_csv(file_in)
        
        # Find rows where y_label == 1
        row_lf_sig = data_train.index[data_train[y_label] == 1]
        # Extract corresponding rows and append to the list
        x_lf_list_sig.append(data_train.loc[row_lf_sig][x_labels].to_numpy())
        # Find rows where y_label == 1
        row_lf = data_train.index[data_train[y_label] < 5]
        # Extract corresponding rows and append to the list
        x_lf_list.append(data_train.loc[row_lf][x_labels].to_numpy())
        y_lf_list.append(data_train.loc[row_lf][y_label].to_numpy())

    except FileNotFoundError:
        print(f"File not found: {file_in}")
    except Exception as e:
        print(f"Error processing file {file_in}: {e}")

# Combine all rows into a single array
x_lf = np.vstack(x_lf_list)
y_lf = np.vstack(y_lf_list)
x_lf_sig = np.vstack(x_lf_list_sig)

# Output the result
print(f"Total rows with y(x) = 1: {x_lf.shape[0]}")

100%|██████████| 300/300 [01:40<00:00,  2.98it/s]


Total rows with y(x) = 1: 15000000


In [ ]:


# Initialize and run the importance sampling workflow
is_sampler = ImportanceSampling(x_lf, y_lf, n_clusters=2, n_samples=1000, exploratory_fraction=0.)
estimated_probability = is_sampler.run(bandwidth_p=1.,bandwidth_q=0.5)

# Output results
print(f"Estimated Probability of y=1: {estimated_probability}")
print(f"True Probability of y=1: {np.mean(y_lf == 1)}")